### Luftballons – Lösung Schritt für Schritt

*35. Bundeswettbewerb Informatik (2016), 1. Runde, Junioraufgabe 1*

Wie kommen in jede Packung immer gleich viele Luftballons? Das ist gar nicht so einfach!
Gegenstand dieser Aufgabe ist eine einfache Luftballonverpackungsmaschine (LVM). Sie hat 10 Speicherfächer sowie ein
Ausgabefach und unterstützt folgende Operationen:

- FACH(i): Entleert das i-te Speicherfach in das Ausgabefach und füllt das Speicherfach mit einer wechselnden Anzahl von
  Luftballons neu.
- VERPACKEN(): Entleert das Ausgabefach und verpackt die darin enthaltenen Luftballons.

Jede Packung soll möglichst 20 Luftballons enthalten. Weniger Ballons würden zu Reklamationen führen; mehr Ballons sind
kostenträchtig, aber notfalls akzeptabel.

Beispiel: Enthalten die Fächer 6, 7, 9 und 3 die Anzahlen 2, 8, 2 und 8, werden durch die Operationen FACH(6), FACH(7), FACH(9), FACH(3),
VERPACKEN() genau 2 + 8 + 2 + 8 = 20 Luftballons verpackt.

#### Junioraufgabe 1

Schreibe ein Programm, das eine LVM simuliert und so steuert, dass jede Packung mindestens 20 und möglichst genau
20 Luftballons enthält. Gehe davon aus, dass zu Beginn alle Fächer leer sind. Simuliere die Operation FACH(i) mit Hilfe
einer Füllfolge, das ist eine Folge von Zahlen: Die nächste noch nicht benutzte Zahl in dieser Folge gibt an, mit wie vielen
Ballons das i-te Fach nach dem Entleeren neu gefüllt wird.

Unter bundeswettbewerb-informatik.de findest du einige Beispiel-Füllfolgen. Dein Programm soll für jedes Beispiel die
durchgeführten Operationen, die Zahl der produzierten Packungen sowie die Zahl der insgesamt verbrauchten Ballons
übersichtlich ausgeben.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `luftballons1.txt` bis `luftballons7.txt`. Jede Datei ist eine Füllfolge mit einer Zahl pro Zeile.
Zu Beginn sind alle Fächer leer. Die ersten zehn Zahlen füllen die Fächer 1 bis 10, die weiteren dienen zum Nachfüllen. Ist die Füllfolge aufgebraucht, werden Fächer
nur noch geleert und bleiben leer.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Simulation einer Maschine:** Die LVM hat einen Zustand (die Inhalte der Fächer, das Ausgabefach, die Füllfolge) und zwei Operationen, die den Zustand ändern. Das programmieren wir nach.
- **Teilmengensumme (Subset Sum):** Aus 10 Zahlen wählen wir eine Teilmenge aus, deren Summe mindestens 20 und möglichst klein ist. Es gibt nur 2¹⁰ = 1024 Teilmengen,
  Ausprobieren (**Brute Force**) ist also kein Problem.
- **Gierige Entscheidungen mit Neuberechnung:** Wir entleeren nur **ein** Fach (das kleinste der besten Teilmenge) und rechnen danach neu, denn das Nachfüllen ändert die Lage.
- **Dynamische Programmierung (für Fortgeschrittene):** Eine Tabelle sagt, welche Summen sich mit den ersten `i` Fächern bilden lassen. Das ist auch bei sehr vielen Fächern schnell.
- **Warteschlange:** Die Füllfolge wird von vorn abgearbeitet (jede Zahl nur einmal).

**Nützliche Funktionen und Bibliotheken**
- `itertools.combinations(liste, k)` liefert alle Auswahlen von `k` Elementen aus einer Liste.
- `sum`, `min`, `len`, Listen und Wörterbücher (`dict`) für den Zustand der Maschine

## Die Idee

**Die Maschine.** Ihr Zustand: die Inhalte der 10 Fächer, die Zahl der Ballons im Ausgabefach (die „Schale“), die Füllfolge und ein Zeiger, welche Zahl der Füllfolge als nächste dran ist.
- `fach(i)`: Der Inhalt des Fachs `i` kommt in die Schale. Das Fach wird mit der nächsten Zahl der Füllfolge gefüllt (oder bleibt leer, wenn die Füllfolge aufgebraucht ist).
- `verpacken()`: Die Schale wird in eine Packung gefüllt und ist danach leer.

Am Anfang sind alle Fächer leer. Wir rufen `fach(1)` bis `fach(10)` auf. Dabei kommen nur 0 Ballons in die Schale, aber die Fächer werden mit den ersten zehn Zahlen der Folge gefüllt.

**Die Steuerung.** Das Ziel einer Packung ist 20. In der Schale liegen schon `schale` Ballons, es fehlen also `ziel = 20 − schale`.
1. Ist `ziel ≤ 0`, rufen wir `verpacken()` auf.
2. Sonst suchen wir unter den Fächern die **beste Auswahl**: eine Teilmenge, deren Summe mindestens `ziel` und möglichst klein ist (gibt es mehrere, nehmen wir die mit **weniger** Fächern).
3. Aus dieser Auswahl entleeren wir nur das Fach mit dem **kleinsten** Inhalt. Dann beginnt die Überlegung von vorn, denn das Fach wurde ja neu gefüllt, und das kann eine bessere Auswahl ermöglichen.
4. Gibt es keine Auswahl (alle Fächer zusammen haben weniger als `ziel`), entleeren wir ebenfalls das Fach mit dem kleinsten (nicht leeren) Inhalt: Dadurch ist der Rest, der noch fehlt, am größten, und es gibt daher
   mehr Möglichkeiten, ihn zu füllen.
5. Das Programm hört auf, wenn die Füllfolge aufgebraucht ist und die Fächer zusammen nicht mehr für eine Packung reichen.

**Warum nur das kleinste Fach?** Große Fächer bleiben in der Maschine, wo sie später mit anderen zu genau 20 kombiniert werden können. Kleine Fächer lassen sich leicht ergänzen.

## Schritt 1: Die Füllfolge einlesen

Eine Zahl pro Zeile. `read().split()` liefert alle Zahlen als Wörter, `int(...)` macht Zahlen daraus.

In [ ]:
def lies_folge(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        return [int(zahl) for zahl in f.read().split()]


folge = lies_folge('beispieldaten/luftballons1.txt')
print(len(folge), 'Zahlen:', folge)

18 Zahlen: [5, 3, 8, 3, 6, 2, 8, 4, 2, 2, 9, 1, 3, 11, 6, 4, 7, 5]


## Schritt 2: Die Maschine

Wir speichern den Zustand der Maschine in einem Wörterbuch (`dict`) mit vier Einträgen: die Liste `faecher` der zehn Fächer, die `schale` (Ballons im Ausgabefach), die `folge` und die Nummer `pos`
der nächsten Zahl in der Füllfolge. Die Funktionen `fach` und `verpacken` ändern dieses Wörterbuch direkt und geben die Zahl der Ballons zurück, die geleert beziehungsweise verpackt wurden.

Die Fächer nummerieren wir wie in der Aufgabe von 1 bis 10 (intern liegen sie auf den Positionen 0 bis 9 der Liste).

In [ ]:
def neue_maschine(folge):
    maschine = {'faecher': [0] * 10, 'schale': 0, 'folge': folge, 'pos': 0}
    for nummer in range(1, 11):
        fach(maschine, nummer)              # alle Fächer zum ersten Mal füllen
    return maschine


def fach(maschine, nummer):
    inhalt = maschine['faecher'][nummer - 1]
    maschine['schale'] = maschine['schale'] + inhalt            # Inhalt in die Schale
    if maschine['pos'] < len(maschine['folge']):
        maschine['faecher'][nummer - 1] = maschine['folge'][maschine['pos']]
        maschine['pos'] = maschine['pos'] + 1                   # nächste Zahl der Füllfolge
    else:
        maschine['faecher'][nummer - 1] = 0                     # Füllfolge aufgebraucht
    return inhalt


def verpacken(maschine):
    packung = maschine['schale']
    maschine['schale'] = 0
    return packung


m = neue_maschine(folge)
print('Fächer:', m['faecher'], ' Schale:', m['schale'])
print('FACH(2) leert', fach(m, 2), 'Ballons, Fächer:', m['faecher'], ' Schale:', m['schale'])
print('VERPACKEN() verpackt', verpacken(m), 'Ballons, Schale:', m['schale'])

Fächer: [5, 3, 8, 3, 6, 2, 8, 4, 2, 2]  Schale: 0
FACH(2) leert 3 Ballons, Fächer: [5, 9, 8, 3, 6, 2, 8, 4, 2, 2]  Schale: 3
VERPACKEN() verpackt 3 Ballons, Schale: 0


## Schritt 3: Die beste Auswahl von Fächern

`beste_auswahl(faecher, ziel)` probiert alle Teilmengen der Fächer aus, die Ballons enthalten. `itertools.combinations(nummern, k)` liefert alle Auswahlen von `k` Fächernummern.
Wir behalten die Auswahl mit der kleinsten Summe, die mindestens `ziel` ist. Bei gleicher Summe gewinnt die Auswahl mit weniger Fächern, und weil wir mit kleinen `k` beginnen,
muss eine spätere nur bei **strikt kleinerer Summe** gewinnen.

Die Funktion gibt die Liste der Fächernummern (1 bis 10) zurück oder die leere Liste, wenn keine Auswahl das Ziel erreicht.

In [ ]:
import itertools


def beste_auswahl(faecher, ziel):
    nummern = []
    for i in range(10):
        if faecher[i] > 0:
            nummern.append(i + 1)           # nur Fächer mit Ballons

    beste = []
    beste_summe = None
    for k in range(1, len(nummern) + 1):
        for auswahl in itertools.combinations(nummern, k):
            summe = 0
            for nummer in auswahl:
                summe = summe + faecher[nummer - 1]
            if summe >= ziel and (beste_summe is None or summe < beste_summe):
                beste = list(auswahl)
                beste_summe = summe
    return beste


f = [5, 3, 8, 3, 6, 2, 8, 4, 2, 2]
auswahl = beste_auswahl(f, 20)
print(auswahl, [f[n - 1] for n in auswahl])
print(beste_auswahl([5, 3, 2, 0, 0, 0, 0, 0, 0, 0], 20))    # reicht nicht: leere Liste

[3, 7, 8] [8, 8, 4]
[]


Für die Fächer `[5, 3, 8, 3, 6, 2, 8, 4, 2, 2]` findet die Funktion eine Auswahl mit genau 20 Ballons aus möglichst wenigen Fächern: die Fächer 3, 7 und 8 mit 8, 8 und 4 Ballons.
Zwei Fächer reichen nicht (8 + 8 = 16), also sind es drei.

## Schritt 4: Die Steuerung

Jetzt die Schleife aus der Idee. Wir schreiben jede Operation in eine Liste `protokoll` (als Text) und zählen die Packungen. In `packungen` sammeln wir die Größen der Packungen.

- Am Anfang sind alle Fächer gefüllt, die Schale ist leer.
- In jedem Durchgang: `ziel = 20 − schale`. Ist `ziel ≤ 0`: verpacken.
- Sonst Auswahl suchen. Gibt es keine Auswahl: das kleinste nicht leere Fach nehmen. Reicht die Füllfolge nicht mehr (alle Fächer zusammen weniger als `ziel` und die Füllfolge ist aufgebraucht), hören wir auf.
- Aus der Auswahl das Fach mit dem kleinsten Inhalt entleeren.

Am Ende bleiben die Ballons, die nicht mehr für eine volle Packung reichen (in den Fächern und in der Schale). Das ist der **Rest**.

In [ ]:
def steuere(folge):
    maschine = neue_maschine(folge)
    protokoll = []
    packungen = []
    while True:
        ziel = 20 - maschine['schale']
        if ziel <= 0:
            groesse = verpacken(maschine)
            protokoll.append(f'VERPACKEN(): {groesse} Luftballons verpackt')
            packungen.append(groesse)
            continue

        faecher = maschine['faecher']
        auswahl = beste_auswahl(faecher, ziel)
        if len(auswahl) == 0:
            # Keine Auswahl erreicht das Ziel: das kleinste nicht leere Fach nehmen (falls es noch Nachschub gibt)
            if maschine['pos'] >= len(folge):
                break                       # Füllfolge aufgebraucht und der Rest reicht nicht mehr
            kandidaten = []
            for i in range(10):
                if faecher[i] > 0:
                    kandidaten.append(i + 1)
            if len(kandidaten) == 0:
                break
            auswahl = kandidaten
        # Das Fach mit dem kleinsten Inhalt aus der Auswahl
        kleinstes = auswahl[0]
        for nummer in auswahl:
            if faecher[nummer - 1] < faecher[kleinstes - 1]:
                kleinstes = nummer
        inhalt = fach(maschine, kleinstes)
        protokoll.append(f'FACH({kleinstes}): {inhalt} Ballons in die Schale, jetzt {maschine["schale"]}')

    rest = sum(maschine['faecher']) + maschine['schale']
    return protokoll, packungen, rest


protokoll, packungen, rest = steuere(lies_folge('beispieldaten/luftballons1.txt'))
for zeile in protokoll[:14]:
    print(zeile)
print('...')
print('Packungen:', len(packungen), ' verpackt insgesamt:', sum(packungen), ' Rest:', rest)

FACH(8): 4 Ballons in die Schale, jetzt 4
FACH(3): 8 Ballons in die Schale, jetzt 12
FACH(7): 8 Ballons in die Schale, jetzt 20
VERPACKEN(): 20 Luftballons verpackt
FACH(1): 5 Ballons in die Schale, jetzt 5
FACH(5): 6 Ballons in die Schale, jetzt 11
FACH(8): 9 Ballons in die Schale, jetzt 20
VERPACKEN(): 20 Luftballons verpackt
FACH(2): 3 Ballons in die Schale, jetzt 3
FACH(5): 6 Ballons in die Schale, jetzt 9
FACH(1): 11 Ballons in die Schale, jetzt 20
VERPACKEN(): 20 Luftballons verpackt
FACH(3): 1 Ballons in die Schale, jetzt 1
FACH(4): 3 Ballons in die Schale, jetzt 4
...
Packungen: 4  verpackt insgesamt: 80  Rest: 9


Die erste Packung besteht aus den Fächern 8, 3 und 7 (4 + 8 + 8 = 20 Ballons). Zuerst wird das Fach mit dem kleinsten Inhalt der besten Auswahl geleert (Fach 8 mit 4 Ballons).
Danach rechnet das Programm neu, denn Fach 8 wurde neu gefüllt (mit 9 Ballons), und das kann eine noch bessere Auswahl ergeben. Hier bleibt die Auswahl dieselbe, und die Fächer 3 und 7 folgen.

## Schritt 5: Alle Beispiele

Für jedes Beispiel geben wir aus, wie viele Packungen welcher Größe entstanden sind, wie viele Ballons verpackt wurden und wie viele übrig blieben. Die ganze Folge der Operationen
(`protokoll`) steht in der Variablen, für die Dokumentation kann man sie in eine Datei schreiben.

In [ ]:
def bearbeite(nr):
    folge = lies_folge(f'beispieldaten/luftballons{nr}.txt')
    protokoll, packungen, rest = steuere(folge)
    groessen = {}
    for p in packungen:
        groessen[p] = groessen.get(p, 0) + 1
    teile = []
    for g in sorted(groessen):
        teile.append(f'{groessen[g]} x {g}')
    operationen = len(protokoll)
    print(f'luftballons{nr}.txt: {len(folge)} Zahlen in der Füllfolge, {operationen} Operationen')
    print(f'  Packungen: {len(packungen)} ({", ".join(teile)}), verpackt: {sum(packungen)} Ballons, Rest: {rest}')


for nr in range(1, 8):
    bearbeite(nr)

luftballons1.txt: 18 Zahlen in der Füllfolge, 18 Operationen
  Packungen: 4 (4 x 20), verpackt: 80 Ballons, Rest: 9
luftballons2.txt: 17 Zahlen in der Füllfolge, 21 Operationen
  Packungen: 4 (4 x 20), verpackt: 80 Ballons, Rest: 0
luftballons3.txt: 100 Zahlen in der Füllfolge, 116 Operationen
  Packungen: 24 (24 x 20), verpackt: 480 Ballons, Rest: 11
luftballons4.txt: 100 Zahlen in der Füllfolge, 118 Operationen
  Packungen: 22 (22 x 20), verpackt: 440 Ballons, Rest: 18
luftballons5.txt: 35 Zahlen in der Füllfolge, 51 Operationen
  Packungen: 17 (9 x 20, 8 x 30), verpackt: 420 Ballons, Rest: 15
luftballons6.txt: 35 Zahlen in der Füllfolge, 51 Operationen
  Packungen: 17 (9 x 20, 8 x 30), verpackt: 420 Ballons, Rest: 15
luftballons7.txt: 100 Zahlen in der Füllfolge, 105 Operationen
  Packungen: 5 (5 x 20), verpackt: 100 Ballons, Rest: 0


Das sind genau die Ergebnisse der offiziellen Lösungshinweise:

| Beispiel | Packungen | verpackt | Rest |
|---|---|---|---|
| 1 | 4 × 20 | 80 | 9 |
| 2 | 4 × 20 | 80 | 0 |
| 3 | 24 × 20 | 480 | 11 |
| 4 | 22 × 20 | 440 | 18 |
| 5 und 6 | 9 × 20 und 8 × 30 | 420 | 15 |
| 7 | 5 × 20 | 100 | 0 |

Bei den Beispielen 5 und 6 (Fächer mit 5 und 15 Ballons) kommt es auf die Reihenfolge an. Würde man zuerst viermal 5 zusammenpacken, wären drei perfekte 20er-Packungen weniger möglich.
Hier entstehen zuerst neun 20er-Packungen und danach acht 30er. Die 30er-Packungen sind nicht zu vermeiden, weil die 15er-Ballons nur paarweise passen.
Bei Beispiel 7 (nur Fächer mit 1 Ballon) werden 20 Fächer geleert, bis die Packung voll ist.

## Für Neugierige: Schneller mit Dynamischer Programmierung

Bei 10 Fächern ist das Durchprobieren aller Teilmengen kein Problem (1024). Eine Maschine mit 30 Fächern hätte über eine Milliarde Teilmengen, Brute Force wäre zu langsam.
Dann hilft eine **Tabelle**: `moeglich[s]` ist wahr, wenn sich die Summe `s` mit den bisher betrachteten Fächern bilden lässt. Für jedes neue Fach mit `inhalt` Ballons gilt:
Alle Summen `s`, die man vorher bilden konnte, kann man jetzt auch mit `inhalt` mehr bilden. Wir müssen nur Summen bis zu einer Obergrenze betrachten (die Summe aller Fächer, oder
nur bis zu `ziel + größtes Fach`).

Die Funktion `kleinste_summe(faecher, ziel)` liefert die kleinste erreichbare Summe, die mindestens `ziel` ist (oder `None`). Wir vergleichen sie mit der Brute-Force-Suche an vielen Zufallsfächern.

In [ ]:
import random


def kleinste_summe(faecher, ziel):
    grenze = sum(faecher)
    moeglich = [False] * (grenze + 1)
    moeglich[0] = True
    for inhalt in faecher:
        if inhalt > 0:
            for s in range(grenze, inhalt - 1, -1):         # rückwärts, damit jedes Fach nur einmal zählt
                if moeglich[s - inhalt]:
                    moeglich[s] = True
    for s in range(max(ziel, 1), grenze + 1):
        if moeglich[s]:
            return s
    return None


def kleinste_summe_brute_force(faecher, ziel):
    auswahl = beste_auswahl(faecher, ziel)
    if len(auswahl) == 0:
        return None
    summe = 0
    for nummer in auswahl:
        summe = summe + faecher[nummer - 1]
    return summe


random.seed(5)
gleich = True
for versuch in range(500):
    faecher = []
    for i in range(10):
        faecher.append(random.randint(0, 12))
    ziel = random.randint(1, 25)
    if kleinste_summe(faecher, ziel) != kleinste_summe_brute_force(faecher, ziel):
        gleich = False
print('Tabelle und Brute Force liefern immer dieselbe kleinste Summe:', gleich)

Tabelle und Brute Force liefern immer dieselbe kleinste Summe: True
